# BMD-45 Per-Folder Loop (cumulative chain + rehearsal replay)

One folder per loop, end to end: prep → train → val → export → package →
ledger. Weights chain forward (`best.pt` of loop N seeds loop N+1), so the
final loop's checkpoint IS the combined model — no ensembling.
Forgetting is held off by rehearsal: each loop trains on the new folder
+ 15% stratified rehearsal JPGs from past folders (cheap: all JPGs ≈ 18 GB).

## Loop 1 vs loops 2+
- Loop 1 (`FOLDER` = first folder, `PREV_WEIGHTS = None`, 20 epochs):
  method validation. If mapping/augments/bicycle look wrong HERE, fix the
  recipe before spending further loops.
- Loops 2+ (`PREV_WEIGHTS` = prior `best.pt`, 12 epochs): accumulate.

## Fixed ruler
Every loop validates on the same official-val anchor (`VAL_FOLDERS`, one
folder ≈ 3.4k images). Deltas across loops are signal, not noise. Full
10k val only for the final report number.

## Steer rule (apply at the ledger output of every loop)
Overall within −0.02 of best-so-far AND `auto` holding → next folder.
Otherwise STOP and adjust recipe (loss weights, mapping, augments).

## Runtime
T4 GPU. One loop ≈ 1.5–2h — one free-tier session per loop is fine.
A dead session loses at most one loop (prep resumes, ledger persists via
download). Keep the tab focused; run cells staged, not Run-all.

## 0. Params — the ONLY cell you edit per loop

In [1]:
# ---- edit per loop -------------------------------------------------
FOLDER = "images_005"  # this loop's train folder (images_000 .. images_007)
PREV_WEIGHTS = "/content/best_f004.pt"   # loop 1: None (= COCO yolov8n.pt). loops 2+: prior best.pt path
EPOCHS = 12           # loop 1: 20. loops 2+: 12
# ---- stable across loops -------------------------------------------
HF_REPO = "iisc-aim/BMD-45"
HF_TRAIN = "BMD-45-Train"
HF_VAL = "BMD-45-Val"
VAL_FOLDERS = ["images_000"]  # fixed val anchor, all loops
# Rehearsal is automatic: prep accumulates JPGs, so each loop trains on
# this folder + ALL past folders (full replay, anti-forgetting).
ROOT = "/tmp/bmd_loop"  # contract + registry + ledger (ephemeral)

TAG = FOLDER  # registry/ledger tag for this loop
print('loop folder:', TAG, '| init:', PREV_WEIGHTS or 'COCO yolov8n.pt', '| epochs:', EPOCHS)

loop folder: images_005 | init: /content/best_f004.pt | epochs: 12


## 1. Setup

Pinned deps, HF auth (Colab Secrets `HF_TOKEN`, anonymous fallback), disk guard.

In [2]:
!pip install -q ultralytics onnx onnxruntime huggingface_hub

import os
assert 'FOLDER' in dir(), 'run cell 0 (Params) first'

try:
    from google.colab import userdata
    _tok = userdata.get('HF_TOKEN')
    if _tok:
        os.environ['HF_TOKEN'] = _tok
        print('HF auth: token loaded')
    else:
        print('HF auth: anonymous (slower)')
except Exception as e:
    print('HF auth: userdata unavailable (%s), anonymous' % e)

import os as _os
# Native high-performance transport (successor to legacy hf_transfer).
# Must be set before huggingface_hub is first imported in the session.
_os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
import shutil
free_gb = shutil.disk_usage('/tmp').free / 1e9
print('/tmp free: %.1f GB' % free_gb)
assert free_gb > 30, 'need ~30 GB free for one loop; free space and retry'
os.makedirs(ROOT, exist_ok=True)


HF auth: userdata unavailable (No module named 'google.colab'), anonymous
/tmp free: 15.1 GB


AssertionError: need ~30 GB free for one loop; free space and retry

## 2. Prep (idempotent — resume-safe)

Downloads + transcodes ONLY missing folders (contract JPGs already on disk are
skipped), parses both COCO JSONs once, and guarantees the fixed val anchor is
present. Re-running after a disconnect resumes instead of restarting.

In [ ]:
assert 'FOLDER' in dir(), 'run cell 0 (Params) first'
import glob, json, os, shutil
os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")
import cv2
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from huggingface_hub import snapshot_download
# Download concurrency. Maximum is NOT optimal here: the xet
# token-refresh endpoint 429s even at 2 workers, so extra workers convert
# to backoff waits, not speed. 4 + retry wrapper is the measured sweet spot
# (raise only with a paid HF tier / dedicated endpoint).
DL_WORKERS = 4
import time as _time, random as _random

def _transient(exc):
    msg = str(exc)
    return ("429" in msg or "Too Many Requests" in msg
            or "ConnectionError" in type(exc).__name__
            or "Timeout" in type(exc).__name__)

def _dl(label, tries=8, base_wait=30.0, **kw):
    # Retry transient (429/connection) failures with exponential backoff.
    # HF caches finished files, so every retry resumes further. Fatal errors
    # raise immediately instead of burning ~10 min of waits.
    for attempt in range(1, tries + 1):
        try:
            return snapshot_download(**kw)
        except Exception as e:
            if not _transient(e) or attempt == tries:
                print("[%s] giving up after %d attempts: %s" % (label, attempt, e))
                raise
            wait = base_wait * (1.6 ** (attempt - 1)) + _random.uniform(0, 10)
            print("[%s] transient (%s), retry %d/%d in %.0fs" % (label, e, attempt + 1, tries, wait))
            _time.sleep(wait)


CLASSES = ["car", "motorcycle", "bus", "truck", "bicycle", "auto"]
ALIASES = {
    "motorbike": "motorcycle", "moto": "motorcycle",
    "two_wheeler": "motorcycle", "two-wheeler": "motorcycle",
    "bicycle": "bicycle", "bike": "bicycle", "cycle": "bicycle",
    "autorickshaw": "auto", "rickshaw": "auto",
    "three_wheeler": "auto", "auto_rickshaw": "auto",
    "three-wheeler": "auto",
    "sedan": "car", "hatchback": "car", "suv": "car", "muv": "car",
    "minibus": "bus", "mini_bus": "bus", "van": "bus",
    "tempo_traveller": "bus", "tempo": "bus",
    "lcv": "truck",
}

def class_index(name):
    n = (name or "").lower().strip().replace(" ", "_").replace("-", "_")
    m = ALIASES.get(n, n)
    return CLASSES.index(m) if m in CLASSES else None

meta = _dl("meta-json", repo_id=HF_REPO, repo_type='dataset',
                allow_patterns=["BMD-45-Train/*.json", "BMD-45-Val/*.json"],
                max_workers=DL_WORKERS)
TABLE = {}  # (split, stem) -> [yolo lines]
for subset, split in ((HF_TRAIN, 'train'), (HF_VAL, 'val')):
    afs = sorted(Path(meta, subset).rglob('*annotations*.json'))
    assert afs, 'no COCO JSON for ' + subset
    data = json.loads(afs[0].read_text(encoding='utf-8'))
    cats = {c['id']: c['name'] for c in data.get('categories', [])}
    imgs = {im['id']: im for im in data.get('images', [])}
    n_keep = n_drop = 0
    for a in data.get('annotations', []):
        cname = cats.get(a.get('category_id'))
        idx = class_index(cname) if cname else None
        if idx is None:
            n_drop += 1
            continue
        im = imgs.get(a.get('image_id'))
        if im is None:
            continue
        iw, ih = im.get('width') or 1, im.get('height') or 1
        x, y, w, h = a['bbox']
        cx, cy = (x + w / 2) / iw, (y + h / 2) / ih
        stem = Path(str(im.get('file_name', ''))).stem
        line = str(idx) + ' %.6f %.6f %.6f %.6f' % (cx, cy, w / iw, h / ih)
        TABLE.setdefault((split, stem), []).append(line)
        n_keep += 1
    print('%s: kept %d boxes, dropped %d' % (subset, n_keep, n_drop))
assert TABLE, 'empty label table'

def ensure_folder(subset, fld, split):
    """Download one folder, transcode its kept images, delete PNGs."""
    have = {p.stem for p in Path(ROOT, 'images', split).glob('*.jpg')} if Path(ROOT, 'images', split).is_dir() else set()
    d = _dl(subset + '/' + fld, repo_id=HF_REPO, repo_type='dataset',
                allow_patterns=[subset + '/' + fld + '/**'],
                max_workers=DL_WORKERS)
    src_dir = Path(d) / subset / fld
    for sub in ('images', 'labels'):
        os.makedirs(os.path.join(ROOT, sub, split), exist_ok=True)
    cands = [p for p in sorted(src_dir.glob('*.png'))
             if TABLE.get((split, p.stem)) and p.stem not in have]

    def _one(png):
        img = cv2.imread(str(png))
        assert img is not None, 'unreadable ' + str(png)
        cv2.imwrite(os.path.join(ROOT, 'images', split, png.stem + '.jpg'),
                    img, [cv2.IMWRITE_JPEG_QUALITY, 92])
        fh = open(os.path.join(ROOT, 'labels', split, png.stem + '.txt'), 'w')
        fh.write('\n'.join(TABLE[(split, png.stem)]) + '\n')
        fh.close()
        return 1

    # threads (not processes): cv2 imread/imwrite release the GIL, and threads
    # avoid re-parsing the multi-GB COCO tables per worker. 4 = free-tier sweet
    # spot; I/O-bound beyond that.
    new = 0
    if cands:
        with ThreadPoolExecutor(max_workers=4) as ex:
            for _ in ex.map(_one, cands):
                new += 1
    shutil.rmtree(src_dir, ignore_errors=True)
    print('done %s/%s: %d new images' % (subset, fld, new))
    return new

ensure_folder(HF_TRAIN, FOLDER, 'train')
for _vf in VAL_FOLDERS:
    ensure_folder(HF_VAL, _vf, 'val')
names = ['path: ' + os.path.abspath(ROOT), 'train: images/train',
         'val: images/val', 'test: images/val', 'names:']
names += ['  %d: %s' % (i, n) for i, n in enumerate(CLASSES)]
open(os.path.join(ROOT, 'data.yaml'), 'w').write('\n'.join(names) + '\n')
ntr = len(list(Path(ROOT, 'images', 'train').glob('*.jpg')))
nva = len(list(Path(ROOT, 'images', 'val').glob('*.jpg')))
print('contract: train %d images, val %d images' % (ntr, nva))
assert ntr > 0 and nva > 0, 'empty split produced'


## 3. Train (warm-start chain)

Loop 1 starts from COCO `yolov8n.pt`; loops 2+ from the prior `best.pt`.
The train dir accumulates JPGs across loops, so every loop trains on this
folder + full rehearsal of all past folders (anti-forgetting, no extra code).

In [ ]:
assert 'FOLDER' in dir(), 'run cell 0 (Params) first'
import os, random
from pathlib import Path as _P
from ultralytics import YOLO

# Train dir ACCUMULATES across loops (prep only adds, never deletes), so
# training on the whole dir = this folder + full rehearsal of all past
# folders. That plus warm-start weights = cumulative chain with replay.
all_train = sorted(_P(ROOT, 'images', 'train').glob('*.jpg'))
print('train pool: %d images (this + all past folders)' % len(all_train))
assert all_train, 'empty train dir — run cell 2 (Prep) first'
init_w = PREV_WEIGHTS or 'yolov8n.pt'
print('init weights:', init_w)
model = YOLO(init_w)
model.train(
    data=os.path.join(ROOT, 'data.yaml'),
    epochs=EPOCHS, imgsz=640, batch=32, patience=8, workers=4,
    name='loop_' + TAG,
)  # GPU OOM? batch=16. Host-RAM kill? workers=2.

run_dir = model.trainer.save_dir
BEST = os.path.join(run_dir, 'weights', 'best.pt')
print('best.pt:', BEST)


## 4. Validate on the fixed anchor + per-class table

In [ ]:
assert 'BEST' in dir(), 'run cell 3 (Train) first'
import os
from ultralytics import YOLO

metrics = YOLO(BEST).val(data=os.path.join(ROOT, 'data.yaml'), verbose=False)
names = metrics.names
print('%-10s %9s %9s %9s' % ('class', 'precision', 'recall', 'mAP50'))
per_class = {}
for i, ci in enumerate(metrics.box.ap_class_index):
    nm = names[int(ci)]
    m = round(float(metrics.box.ap50[i]), 4)
    per_class[nm] = m
    print('%-10s %9.3f %9.3f %9.3f' % (nm, float(metrics.box.p[i]), float(metrics.box.r[i]), m))
mAP50 = round(float(metrics.box.map50), 4)
print('overall mAP50: %.3f' % mAP50)


## 5. Export + static int8 + per-loop registry

In [ ]:
assert 'BEST' in dir(), 'run cell 3 (Train) first'
import glob, os
import shutil as _sh, traceback
import cv2 as _cv2, numpy as _np
from onnxruntime.quantization import CalibrationDataReader as _CDR, quantize_static as _qs, QuantType as _QT

onnx_path = YOLO(BEST).export(format='onnx', imgsz=640, opset=17, simplify=True)
print('onnx:', onnx_path)

calib_imgs = sorted(glob.glob(os.path.join(ROOT, 'images', 'val', '*.jpg')))[:200]
assert calib_imgs, 'no val images for calibration'

class _FR(_CDR):
    def __init__(self, frames, onx):
        import onnxruntime as _ort
        self.input_name = _ort.InferenceSession(onx, providers=['CPUExecutionProvider']).get_inputs()[0].name
        self.reiter = iter([self._pre(p) for p in frames])
    def _pre(self, path):
        img = _cv2.imread(path)
        img = _cv2.resize(img, (640, 640))
        rgb = _cv2.cvtColor(img, _cv2.COLOR_BGR2RGB).astype(_np.float32) / 255.0
        return {self.input_name: _np.transpose(rgb, (2, 0, 1))[_np.newaxis]}
    def get_next(self):
        return next(self.reiter, None)

int8_path = onnx_path.replace('.onnx', '-int8.onnx')
try:
    _qs(onnx_path, int8_path, _FR(calib_imgs, onnx_path), weight_type=_QT.QInt8)
    print('STATIC int8 ->', int8_path)
except Exception:
    traceback.print_exc()
    print('STATIC failed: re-run this cell (frames may still be copying)')
    raise

import datetime, json
reg = os.path.join(ROOT, 'registry', 'india-yolov8n-' + TAG)
os.makedirs(reg, exist_ok=True)
_sh.copy(onnx_path, os.path.join(reg, 'model.onnx'))
_sh.copy(int8_path, os.path.join(reg, 'model-int8.onnx'))
meta = {'name': 'india-yolov8n-' + TAG, 'classes': CLASSES, 'imgsz': 640,
        'normalization': {'mean': [0, 0, 0], 'std': [255, 255, 255], 'layout': 'NCHW', 'color': 'RGB'},
        'quantization': 'int8', 'source_run': datetime.date.today().isoformat(),
        'metrics': {'mAP50_overall': mAP50, 'per_class_mAP50': per_class},
        'provenance': {'hf_repo': HF_REPO, 'hf_subset': HF_TRAIN, 'folder': TAG,
                       'init_weights': init_w, 'epochs': EPOCHS,
                       'mapping': 'optionB-merge-6class', 'val': 'official-anchor'}}
open(os.path.join(reg, 'metadata.json'), 'w').write(json.dumps(meta, indent=2))
print('registry:', reg)


## 6. Ledger (steering trail — download every loop)

Append + print + download. Apply the steer rule HERE before the next folder:
overall within −0.02 of best AND `auto` holding → continue; else stop.

In [ ]:
assert 'mAP50' in dir() and 'BEST' in dir(), 'run cells 3-4 first'
import os
import csv
from google.colab import files

ledger = os.path.join(ROOT, 'loop_ledger.csv')
new_file = not os.path.exists(ledger)
fh = open(ledger, 'a', newline='')
wr = csv.writer(fh)
if new_file:
    wr.writerow(['folder', 'init', 'epochs', 'overall', 'auto', 'motorcycle', 'truck', 'bicycle', 'car', 'bus', 'best_pt'])
wr.writerow([TAG, init_w, EPOCHS, mAP50] + [per_class.get(k) for k in ('auto', 'motorcycle', 'truck', 'bicycle', 'car', 'bus')] + [BEST])
fh.close()
print(open(ledger).read())
files.download(ledger)
files.download(os.path.join(ROOT, 'registry', 'india-yolov8n-' + TAG, 'metadata.json'))
print('NEXT LOOP: set FOLDER=<next>, PREV_WEIGHTS=<this best.pt path above>, EPOCHS=12, re-run from cell 0.')
print('(best.pt itself stays in /tmp — for multi-session chains, download it via files.download(BEST) too.)')

## 7. Loop output pack (auto-download — drop this zip in the workspace)

Zips everything needed for review into one file named
`bmd_loop_<folder>_<date>.zip`: the loop registry (onnx + int8 + metadata),
the ledger CSV, `best.pt`, and the training `results.csv`. Download it and
place it in the workspace — no hunting across cells.

In [ ]:
assert 'BEST' in dir() and 'mAP50' in dir(), 'run cells 3-6 first'
import datetime, os, shutil, zipfile

pack_name = 'bmd_loop_%s_%s' % (TAG, datetime.date.today().isoformat())
pack_dir = os.path.join(ROOT, pack_name)
os.makedirs(pack_dir, exist_ok=True)

# registry (onnx + int8 + metadata.json)
shutil.copytree(os.path.join(ROOT, 'registry', 'india-yolov8n-' + TAG),
                os.path.join(pack_dir, 'registry'),
                dirs_exist_ok=True)
# ledger
shutil.copy2(os.path.join(ROOT, 'loop_ledger.csv'), pack_dir)
# best.pt + training curve
shutil.copy2(BEST, os.path.join(pack_dir, 'best.pt'))
results_csv = os.path.join(os.path.dirname(BEST), '..', 'results.csv')
if os.path.isfile(results_csv):
    shutil.copy2(results_csv, pack_dir)

zip_path = shutil.make_archive(os.path.join(ROOT, pack_name), 'zip', root_dir=ROOT, base_dir=pack_name)
size_mb = os.path.getsize(zip_path) / 1e6
print('pack: %s (%.1f MB)' % (zip_path, size_mb))
assert size_mb < 1900, 'pack implausibly large, aborting download'

from google.colab import files
files.download(zip_path)
print('DONE loop %s — place %s.zip in the workspace.' % (TAG, pack_name))
